# QuickBite Online Food Delivery
## Part 1: Cover page

| Field | Details |
|---|---|
| Student name | Aadit Rangnekar |
| Roll number | 48 |
| Odd / Even | Even |
| Case study | B - QuickBite Online Food Delivery |
| Database | MySQL |
| Date of submission | 21 October 2026 |

## Part 2: Schema design

Customers, restaurants and delivery agents have many orders. Restaurants offer
menu items, and OrderItems links them to orders. Each order has one payment
and an optional review; reviews also reference customers.


### Customers

| Column | Data type | Constraints |
|---|---|---|
| customer_id | INT | PRIMARY KEY |
| customer_name | VARCHAR(100) | NOT NULL |
| email | VARCHAR(255) | NOT NULL, UNIQUE |
| phone | VARCHAR(20) | NOT NULL |
| city | VARCHAR(100) | NOT NULL |
| join_date | DATE | NOT NULL |

### Restaurants

| Column | Data type | Constraints |
|---|---|---|
| restaurant_id | INT | PRIMARY KEY |
| restaurant_name | VARCHAR(100) | NOT NULL |
| city | VARCHAR(100) | NOT NULL |
| cuisine_type | VARCHAR(100) | NOT NULL |
| avg_rating | DECIMAL(3,2) | NOT NULL, CHECK 0 to 5 |

### MenuItems

| Column | Data type | Constraints |
|---|---|---|
| item_id | INT | PRIMARY KEY |
| restaurant_id | INT | NOT NULL, FK to Restaurants.restaurant_id |
| item_name | VARCHAR(100) | NOT NULL |
| category | VARCHAR(20) | NOT NULL, CHECK Veg / Non-Veg |
| price | DECIMAL(10,2) | NOT NULL, CHECK > 0 |

### DeliveryAgents

| Column | Data type | Constraints |
|---|---|---|
| agent_id | INT | PRIMARY KEY |
| agent_name | VARCHAR(100) | NOT NULL |
| phone | VARCHAR(20) | NOT NULL |
| vehicle_type | VARCHAR(100) | NOT NULL |
| city | VARCHAR(100) | NOT NULL |

### Orders

| Column | Data type | Constraints |
|---|---|---|
| order_id | INT | PRIMARY KEY |
| customer_id | INT | NOT NULL, FK to Customers.customer_id |
| restaurant_id | INT | NOT NULL, FK to Restaurants.restaurant_id |
| agent_id | INT | NOT NULL, FK to DeliveryAgents.agent_id |
| order_date | DATETIME | NOT NULL |
| order_status | VARCHAR(20) | NOT NULL, CHECK Placed / Delivered / Cancelled |

### OrderItems

| Column | Data type | Constraints |
|---|---|---|
| order_item_id | INT | PRIMARY KEY |
| order_id | INT | NOT NULL, FK to Orders.order_id |
| item_id | INT | NOT NULL, FK to MenuItems.item_id |
| quantity | INT | NOT NULL, CHECK > 0 |

Table constraint: UNIQUE (order_id, item_id).

### Payments

| Column | Data type | Constraints |
|---|---|---|
| payment_id | INT | PRIMARY KEY |
| order_id | INT | NOT NULL, UNIQUE, FK to Orders.order_id |
| payment_mode | VARCHAR(20) | NOT NULL, CHECK UPI / Card / Cash / Wallet |
| amount | DECIMAL(10,2) | NOT NULL, CHECK >= 0 |
| payment_status | VARCHAR(20) | NOT NULL, CHECK Success / Failed |

### Reviews

| Column | Data type | Constraints |
|---|---|---|
| review_id | INT | PRIMARY KEY |
| order_id | INT | NOT NULL, UNIQUE, FK to Orders.order_id |
| customer_id | INT | NOT NULL, FK to Customers.customer_id |
| rating | INT | NOT NULL, CHECK 1 to 5 |
| comments | TEXT | Optional |

## Part 3: Python-SQL connectivity

In [1]:
import mysql.connector
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv(".env", override=True)
database = os.getenv("MYSQL_DATABASE", "quickbite_roll48")
conn = mysql.connector.connect(
    host=os.getenv("MYSQL_HOST", "127.0.0.1"),
    port=int(os.getenv("MYSQL_PORT", "3306")),
    user=os.getenv("MYSQL_USER", "root"),
    password=os.environ["MYSQL_PASSWORD"], connection_timeout=5
)
cursor = conn.cursor()
cursor.execute(f"CREATE DATABASE IF NOT EXISTS {database} CHARACTER SET utf8mb4")
cursor.execute(f"USE {database}")
print(f"Connected to MySQL {conn.server_info}; database: {database}")

def show(sql):
    cursor.execute(sql)
    return pd.DataFrame(cursor.fetchall(), columns=cursor.column_names)

Connected to MySQL 9.6.0; database: quickbite_roll48


### Reset database

In [2]:
if database != "quickbite_roll48":
    raise ValueError("Reset is allowed only for quickbite_roll48.")
cursor.execute("DROP DATABASE IF EXISTS quickbite_roll48")
cursor.execute("CREATE DATABASE quickbite_roll48 CHARACTER SET utf8mb4")
cursor.execute("USE quickbite_roll48")
print("Reset quickbite_roll48. Ready to create all 8 tables.")

Reset quickbite_roll48. Ready to create all 8 tables.


## Part 4: Table creation

In [3]:
ddl = {
    "Customers": """
        customer_id INT PRIMARY KEY,
        customer_name VARCHAR(100) NOT NULL,
        email VARCHAR(255) NOT NULL UNIQUE,
        phone VARCHAR(20) NOT NULL, city VARCHAR(100) NOT NULL,
        join_date DATE NOT NULL""",
    "Restaurants": """
        restaurant_id INT PRIMARY KEY,
        restaurant_name VARCHAR(100) NOT NULL,
        city VARCHAR(100) NOT NULL, cuisine_type VARCHAR(100) NOT NULL,
        avg_rating DECIMAL(3,2) NOT NULL CHECK (avg_rating BETWEEN 0 AND 5)""",
    "MenuItems": """
        item_id INT PRIMARY KEY, restaurant_id INT NOT NULL,
        item_name VARCHAR(100) NOT NULL,
        category VARCHAR(20) NOT NULL CHECK (category IN ('Veg', 'Non-Veg')),
        price DECIMAL(10,2) NOT NULL CHECK (price > 0),
        FOREIGN KEY (restaurant_id) REFERENCES Restaurants(restaurant_id)""",
    "DeliveryAgents": """
        agent_id INT PRIMARY KEY, agent_name VARCHAR(100) NOT NULL,
        phone VARCHAR(20) NOT NULL, vehicle_type VARCHAR(100) NOT NULL,
        city VARCHAR(100) NOT NULL""",
    "Orders": """
        order_id INT PRIMARY KEY, customer_id INT NOT NULL,
        restaurant_id INT NOT NULL, agent_id INT NOT NULL,
        order_date DATETIME NOT NULL,
        order_status VARCHAR(20) NOT NULL
            CHECK (order_status IN ('Placed', 'Delivered', 'Cancelled')),
        FOREIGN KEY (customer_id) REFERENCES Customers(customer_id),
        FOREIGN KEY (restaurant_id) REFERENCES Restaurants(restaurant_id),
        FOREIGN KEY (agent_id) REFERENCES DeliveryAgents(agent_id)""",
    "OrderItems": """
        order_item_id INT PRIMARY KEY, order_id INT NOT NULL,
        item_id INT NOT NULL, quantity INT NOT NULL CHECK (quantity > 0),
        UNIQUE (order_id, item_id),
        FOREIGN KEY (order_id) REFERENCES Orders(order_id),
        FOREIGN KEY (item_id) REFERENCES MenuItems(item_id)""",
    "Payments": """
        payment_id INT PRIMARY KEY, order_id INT NOT NULL UNIQUE,
        payment_mode VARCHAR(20) NOT NULL
            CHECK (payment_mode IN ('UPI', 'Card', 'Cash', 'Wallet')),
        amount DECIMAL(10,2) NOT NULL CHECK (amount >= 0),
        payment_status VARCHAR(20) NOT NULL
            CHECK (payment_status IN ('Success', 'Failed')),
        FOREIGN KEY (order_id) REFERENCES Orders(order_id)""",
    "Reviews": """
        review_id INT PRIMARY KEY, order_id INT NOT NULL UNIQUE,
        customer_id INT NOT NULL,
        rating INT NOT NULL CHECK (rating BETWEEN 1 AND 5), comments TEXT,
        FOREIGN KEY (order_id) REFERENCES Orders(order_id),
        FOREIGN KEY (customer_id) REFERENCES Customers(customer_id)"""
}
for table, definition in ddl.items():
    cursor.execute(f"CREATE TABLE IF NOT EXISTS {table} ({definition}) ENGINE=InnoDB")
display(show("SHOW TABLES"))

,Tables_in_quickbite_roll48
0,Customers
1,DeliveryAgents
2,MenuItems
3,OrderItems
4,Orders
5,Payments
6,Restaurants
7,Reviews


## Part 5: Data insertion and table contents

In [4]:
data = {
    "Customers": [
        (1, "Anaya Deshmukh", "anaya48@example.test", "0000004801", "Mumbai", "2026-09-01"),
        (2, "Vihaan Shah", "vihaan48@example.test", "0000004802", "Mumbai", "2026-09-03"),
        (3, "Ira Kulkarni", "ira48@example.test", "0000004803", "Pune", "2026-09-05"),
        (4, "Kabir Patil", "kabir48@example.test", "0000004804", "Nashik", "2026-09-07"),
        (5, "Sana Shaikh", "sana48@example.test", "0000004805", "Nashik", "2026-09-09")],
    "Restaurants": [
        (1, "Harbour Bowl", "Mumbai", "Indian", "4.40"),
        (2, "Ember Slice", "Mumbai", "Italian", "4.20"),
        (3, "Pune Wok", "Pune", "Chinese", "4.30"),
        (4, "Godavari Kitchen", "Nashik", "Maharashtrian", "4.50"),
        (5, "Leaf and Ladle", "Nashik", "South Indian", "4.10")],
    "MenuItems": [
        (1, 1, "Paneer Rice Bowl", "Veg", "200.00"),
        (2, 1, "Chicken Rice Bowl", "Non-Veg", "300.00"),
        (3, 2, "Margherita Pizza", "Veg", "350.00"),
        (4, 3, "Chilli Chicken", "Non-Veg", "280.00"),
        (5, 4, "Misal Pav", "Veg", "150.00"),
        (6, 5, "Masala Dosa", "Veg", "120.00")],
    "DeliveryAgents": [
        (1, "Omkar Jadhav", "0000004811", "Scooter", "Mumbai"),
        (2, "Rhea Naik", "0000004812", "Bike", "Mumbai"),
        (3, "Dev More", "0000004813", "Scooter", "Pune"),
        (4, "Meera Kale", "0000004814", "Bike", "Nashik"),
        (5, "Adil Khan", "0000004815", "Bicycle", "Nashik")],
    "Orders": [
        (1, 1, 1, 1, "2026-10-10 12:15:00", "Delivered"),
        (2, 1, 1, 1, "2026-10-11 19:20:00", "Delivered"),
        (3, 2, 2, 2, "2026-10-12 13:00:00", "Delivered"),
        (4, 2, 2, 2, "2026-10-13 20:10:00", "Delivered"),
        (5, 3, 3, 3, "2026-10-14 18:30:00", "Delivered"),
        (6, 4, 4, 4, "2026-10-15 11:45:00", "Cancelled")],
    "OrderItems": [
        (1, 1, 1, 1), (2, 1, 2, 1), (3, 2, 1, 2), (4, 2, 2, 1),
        (5, 3, 3, 2), (6, 4, 3, 1), (7, 5, 4, 1), (8, 6, 5, 4)],
    "Payments": [
        (1, 1, "Cash", "500.00", "Success"),
        (2, 2, "UPI", "700.00", "Success"),
        (3, 3, "Card", "700.00", "Success"),
        (4, 4, "UPI", "350.00", "Success"),
        (5, 5, "Wallet", "280.00", "Success"),
        (6, 6, "Card", "600.00", "Failed")],
    "Reviews": [
        (1, 1, 1, 4, "Fresh bowl and timely delivery."),
        (2, 2, 1, 5, "Good portions for a shared meal."),
        (3, 3, 2, 4, "Pizza arrived hot."),
        (4, 4, 2, 3, "Good taste; packaging could improve."),
        (5, 5, 3, 5, "Spice level was just right.")]
}

try:
    for table, rows in data.items():
        marks = ", ".join(["%s"] * len(rows[0]))
        cursor.executemany(f"INSERT INTO {table} VALUES ({marks})", rows)
    conn.commit()
    print("Committed 46 rows across 8 tables.")
except Exception:
    conn.rollback()
    raise

Committed 46 rows across 8 tables.


In [5]:
for table in data:
    print(f"{table}: {len(data[table])} rows")
    display(show(f"SELECT * FROM {table} ORDER BY 1"))

Customers: 5 rows


,customer_id,customer_name,email,phone,city,join_date
0,1,Anaya Deshmukh,anaya48@example.test,0000004801,Mumbai,2026-09-01
1,2,Vihaan Shah,vihaan48@example.test,0000004802,Mumbai,2026-09-03
2,3,Ira Kulkarni,ira48@example.test,0000004803,Pune,2026-09-05
3,4,Kabir Patil,kabir48@example.test,0000004804,Nashik,2026-09-07
4,5,Sana Shaikh,sana48@example.test,0000004805,Nashik,2026-09-09


Restaurants: 5 rows


,restaurant_id,restaurant_name,city,cuisine_type,avg_rating
0,1,Harbour Bowl,Mumbai,Indian,4.40
1,2,Ember Slice,Mumbai,Italian,4.20
2,3,Pune Wok,Pune,Chinese,4.30
3,4,Godavari Kitchen,Nashik,Maharashtrian,4.50
4,5,Leaf and Ladle,Nashik,South Indian,4.10


MenuItems: 6 rows


...

,order_item_id,order_id,item_id,quantity
0,1,1,1,1
1,2,1,2,1
2,3,2,1,2
3,4,2,2,1
4,5,3,3,2
5,6,4,3,1
6,7,5,4,1
7,8,6,5,4


Payments: 6 rows


,payment_id,order_id,payment_mode,amount,payment_status
0,1,1,Cash,500.00,Success
1,2,2,UPI,700.00,Success
2,3,3,Card,700.00,Success
3,4,4,UPI,350.00,Success
4,5,5,Wallet,280.00,Success
5,6,6,Card,600.00,Failed


Reviews: 5 rows


,review_id,order_id,customer_id,rating,comments
0,1,1,1,4,Fresh bowl and timely delivery.
1,2,2,1,5,Good portions for a shared meal.
2,3,3,2,4,Pizza arrived hot.
3,4,4,2,3,Good taste; packaging could improve.
4,5,5,3,5,Spice level was just right.


## Part 6: Query execution and results

Amounts are in INR.

### Q1

Show each restaurant's name, the number of menu items it offers and the average price of those items.

Harbour Bowl offers two items; the other restaurants offer one each.

In [6]:
q1 = show("""
SELECT r.restaurant_name, COUNT(m.item_id) AS menu_items,
       ROUND(AVG(m.price), 2) AS average_price
FROM Restaurants r
LEFT JOIN MenuItems m ON r.restaurant_id = m.restaurant_id
GROUP BY r.restaurant_id, r.restaurant_name
ORDER BY r.restaurant_id
""")
display(q1)

,restaurant_name,menu_items,average_price
0,Harbour Bowl,2,250.00
1,Ember Slice,1,350.00
2,Pune Wok,1,280.00
3,Godavari Kitchen,1,150.00
4,Leaf and Ladle,1,120.00


### Q2

Show each payment mode and the total amount collected through it. Display only payment modes whose total amount is more than INR 500.

Only successful payments are counted; totals equal to INR 500 are excluded.

In [7]:
q2 = show("""
SELECT payment_mode, SUM(amount) AS total_collected
FROM Payments
WHERE payment_status = 'Success'
GROUP BY payment_mode
HAVING SUM(amount) > 500
ORDER BY total_collected DESC, payment_mode
""")
display(q2)

,payment_mode,total_collected
0,UPI,1050.00
1,Card,700.00


### Q3

Show each customer's name and the number of orders they placed. Display only customers who placed 2 or more orders.

All order statuses are included.

In [8]:
q3 = show("""
SELECT c.customer_name, COUNT(o.order_id) AS order_count
FROM Customers c
JOIN Orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.customer_name
HAVING COUNT(o.order_id) >= 2
ORDER BY c.customer_id
""")
display(q3)

,customer_name,order_count
0,Anaya Deshmukh,2
1,Vihaan Shah,2


### Q4

Show each delivery agent's name and the number of orders assigned to them.

Agents with zero assignments and cancelled orders are included.

In [9]:
q4 = show("""
SELECT a.agent_name, COUNT(o.order_id) AS assigned_orders
FROM DeliveryAgents a
LEFT JOIN Orders o ON a.agent_id = o.agent_id
GROUP BY a.agent_id, a.agent_name
ORDER BY a.agent_id
""")
display(q4)

,agent_name,assigned_orders
0,Omkar Jadhav,2
1,Rhea Naik,2
2,Dev More,1
3,Meera Kale,1
4,Adil Khan,0


### Q5

Show each menu category (Veg / Non-Veg), the number of items in it and the highest item price.

Veg has four items; Non-Veg has two.

In [10]:
q5 = show("""
SELECT category, COUNT(*) AS item_count, MAX(price) AS highest_price
FROM MenuItems
GROUP BY category
ORDER BY category
""")
display(q5)

,category,item_count,highest_price
0,Non-Veg,2,300.00
1,Veg,4,350.00


### Results summary

In [11]:
results = [q1, q2, q3, q4, q5]
notes = [
    "Harbour Bowl offers 2 items; the other restaurants offer 1 each.",
    "UPI collected INR 1050 and Card INR 700; both exceed INR 500.",
    "Anaya Deshmukh and Vihaan Shah each placed 2 orders.",
    "Omkar and Rhea have 2 assignments each; Adil has none.",
    "Veg has 4 items (maximum INR 350); Non-Veg has 2 (maximum INR 300)."
]
summary = pd.DataFrame([
    (f"Q{i}", result.to_string(index=False), len(result), note)
    for i, (result, note) in enumerate(zip(results, notes), 1)
], columns=["Query", "Output", "Rows returned", "Interpretation"])
display(summary)

,Query,Output,Rows returned,Interpretation
0,Q1,restaurant_name menu_items average_price\n ...,5,Harbour Bowl offers 2 items; the other restaur...
1,Q2,payment_mode total_collected\n UPI ...,2,UPI collected INR 1050 and Card INR 700; both ...
2,Q3,customer_name order_count\nAnaya Deshmukh ...,2,Anaya Deshmukh and Vihaan Shah each placed 2 o...
3,Q4,agent_name assigned_orders\nOmkar Jadhav ...,5,Omkar and Rhea have 2 assignments each; Adil h...
4,Q5,category item_count highest_price\n Non-Veg ...,2,Veg has 4 items (maximum INR 350); Non-Veg has...


In [12]:
cursor.close()
conn.close()
print("MySQL connection closed.")

MySQL connection closed.


## Part 7: Conclusion

The eight tables represent the QuickBite ordering process.

Python was used to create and populate the MySQL database.

Primary and foreign keys keep related records linked.

Joins and GROUP BY summarize menu items and order counts.

HAVING filters payment totals and customer order counts.

UPI has the highest successful payment total at INR 1050.